# Monte Carlo Delta Estimation

This notebook explores Monte Carlo methods for estimating the sensitivity (delta) of option values with respect to the initial stock price. We consider:

- Finite-difference (bump-and-revalue) estimators for a European call option and for a digital-style payoff.
- Pathwise (trajectory-based) estimators for a smoothed digital payoff.
- Likelihood ratio (score function) estimators for the same smoothed digital payoff.

Throughout, we work under a geometric Brownian motion model with constant parameters and focus on how bias, variance, and payoff smoothness influence the behaviour of delta estimators.

**Structure**

1. European call delta via bump-and-revalue (finite differences) and MSE-based bump selection.
2. Digital option delta with and without smoothing.
3. Smoothed digital payoff with pathwise and likelihood-ratio delta estimators and a comparative discussion.

In [1]:
# Imports and model parameters

import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Global model parameters (can be adjusted)
T = 1.0          # time to maturity
N = 50           # number of time steps for Euler discretization
S0 = 100.0       # initial stock price
K = 99.0         # strike
r = 0.06         # risk-free rate
sigma = 0.2      # volatility

M_default = 3000  # default number of Monte Carlo paths

np.random.seed(12345)  # for reproducibility

#  1.1 Theory behind the greek $\Delta$ approximation using the MC estimate

## Delta Definition

The **delta** of an option measures how sensitive the option value is to changes in the initial stock price $S_0$.

Delta is defined as:

$$\Delta = \frac{\partial V}{\partial S_0}$$

Under risk-neutral pricing, the option value is:

$$V(S_0) = e^{-rT} \mathbb{E}[\text{payoff}(S_T)]$$

where $S_T$ depends on both $S_0$ and the random normal variables driving the simulation.

---

## Forward Difference (Bump-and-Revalue)

From basic calculus:

$$f'(x_0) \approx \hat{d}f_x(x_0; h) = \frac{f(x_0 + h) - f(x_0)}{h}$$

Applying this idea to the option value:

$$\Delta \approx \frac{V(S_0 + h) - V(S_0)}{h}$$

Because the terminal stock price depends on randomness, we simulate:

$$S_T = S_T(S_0, Z)$$

where $Z$ represents the random normal variables. The Monte Carlo delta estimator becomes:

$$\hat{\Delta}(h) = \frac{V(S_T(S_0 + h, Z)) - V(S_T(S_0, Z))}{h}$$

Using the **same random numbers** $Z$ in both terms (common random numbers) introduces strong correlation between the two estimates and significantly reduces variance.

---

## Bias Vs Variance of the Finite Differences Method

By a Taylor approximation around $x_0$ we find the bias to be given by

$$\hat{d}f_x(x_0; h) - \frac{df(x_0)}{dx} = \frac{1}{h} \left( f(x_0 + h) - f(x_0) - hf'(x_0) \right)$$

$$= \frac{1}{h} \left( f(x_0) + f'(x_0)h + \frac{1}{2}f''(x_0)h^2 + \mathcal{O}(h^3) - f(x_0) - hf'(x_0) \right)$$

$$= \frac{1}{h} \left( \frac{1}{2}f''(x_0)h^2 + \mathcal{O}(h^3) \right)$$

$$= \mathcal{O}(h).$$


Considering now the variance of the finite difference estimator we end up having approximately:

$$\text{Var}(\hat{d}f_x(x_0; h)) = \frac{\text{Var}(f(x_0 + h)) + \text{Var}(f(x_0)) - 2\text{Cov}(f(x_0 + h), f(x_0))}{h^2}$$

when using independent samples. 

Combining these effects gives us the classical **bias–variance tradeoff**:

- **Small $h$** → small bias but potentially large variance
- **Large $h$** → larger bias but smaller variance

The optimal choice of $h$ balances these two sources of error.